# धडा १३ - एजंट स्मृती


## सेटअप

हा नोटबुक **Microsoft Agent Framework** (MAF) चा वापर करून **सतत स्मृती** असलेला प्रवास बुकिंग एजंट कसा तयार करायचा हे दाखवते.

तुम्हाला कसे वेगवेगळ्या प्रकारच्या एजंट स्मृती — कार्यरत, अल्पकालीन, आणि दीर्घकालीन — एखाद्या एजंट कसा संभाषणादरम्यान माहिती टिकवतो आणि वापरतो यावर परिणाम करतात हे शिकायला मिळेल.

**प्रीरिक्विझिट्स:**
- एक Microsoft Foundry प्रोजेक्ट ज्यामध्ये एक चालू चॅट मॉडेल (उदा. `gpt-5-mini`) असेल.
- Azure CLI मध्ये लॉग इन केलेले — तुमच्या टर्मिनलमध्ये `az login` चालवा.
- `AZURE_AI_PROJECT_ENDPOINT` — तुमचा Microsoft Foundry प्रोजेक्ट एंडपॉइंट.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तुझ्या तैनात केलेल्या मॉडेलचे नाव.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## एजंट मेमरीचे प्रकार

AI एजंट विविध प्रकारच्या मेमरीचा वापर करू शकतात, ज्यापैकी प्रत्येकाचा वेगळा उद्देश असतो:

### कार्यरत मेमरी
संभाषणाचा धागा स्वतः — एका सत्रामध्ये देवाणघेवाण केलेली संदेशे. एजंट त्याच धाग्यातील आधीच्या संदेशांकडे परत पाहू शकतो जेणेकरून सुसंगती राखली जाईल. MAF मध्ये हे **`agent.create_session()`** द्वारे तयार केले जाते, जे `AgentSession` परत करते.

### अल्पकालीन मेमरी
अशी माहिती जी एखाद्या कार्य किंवा सत्राच्या कालावधीसाठी टिकून राहते परंतु कायमस्वरूपी जतन केली जात नाही. उदाहरणार्थ, एजंट बहु-चरण नियोजन संभाषणादरम्यान तथ्ये जमा करू शकतो आणि त्यांचा उपयोग अंतिम प्रवास योजनेसाठी करू शकतो.

### दीर्घकालीन मेमरी
पसंती आणि तथ्ये जी **सत्रानंतरही** टिकून राहतात. परत आलेल्या वापरकर्त्याने त्यांच्या आहार प्रतिबंध किंवा प्रवास शैली परत सांगावी लागणार नाही. दीर्घकालीन मेमरी सहसा बाह्य संचयनाद्वारे समर्थित असते — डेटाबेस, फाइल किंवा व्हेक्टर निर्देशांक — आणि एजंटला उपकरणांद्वारे उपलब्ध होते.


## सत्रांसह कार्यशील स्मृती

स्मृतीचा सर्वात सोपा प्रकार म्हणजे संभाषण सत्र. जेव्हा तुम्ही सारखेच सत्र (`agent.create_session()` द्वारे तयार केलेले) सलग `agent.run()` कॉल्सना पास करता, तेव्हा एजंट त्या संभाषणाचा पूर्ण इतिहास पाहतो आणि आधीच्या तपशीलांची आठवण ठेवू शकतो.

चला एक प्रवासी एजंट तयार करू आणि कार्यशील स्मृती दाखवू.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजंटने बजेट योग्यरित्या आठवले कारण दोन्ही मेसेजेस एकाच सत्रातून होते. हे **कार्यात्मक स्मृती** आहे — ते फक्त सत्राच्या आयुष्यासाठी अस्तित्वात असते.

### नवीन थ्रेड असताना काय होते?

जर आपण **नवीन** सत्र तयार केले, तर एजंटला मागील संभाषणाची कोणतीही स्मृती नसते:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालीन स्मृती नमुना

वापरकर्त्याच्या पसंती **सत्रांदरम्यान** लक्षात ठेवण्यासाठी, आपल्याला एक कायमस्वरूपी संचित आवश्यक आहे जे संभाषण थ्रेडच्या बाहेर राहते. एजंट या संचितापर्यंत **टूल्स** — अशी कार्ये ज्यांना ते माहिती सेव्ह आणि पुनःप्राप्त करण्यासाठी कॉल करू शकते, याद्वारे प्रवेश करतो.

खाली आपण एक सोपी इन-मेमरी प्राधान्य संचय राबवित आहोत (उत्पादनात तुम्ही याला डेटाबेस किंवा व्हेक्टर निर्देशांकाने समर्थित कराल) आणि एजंट वापरू शकणाऱ्या टूल्स म्हणून ते प्रदर्शित करतो.

### आर्किटेक्चर
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### घडामोड 1 — प्रथमच वापरकर्ता वर्धापनदिनाची सहल बुक करतो

सौरा प्रथमच भेट देते. एजंटने तिच्या पसंती साधनांद्वारे संग्रहित कराव्यात आणि त्यांचा वापर करून हॉटेल शिफारस कराव्यात.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिस्थिती 2 — सारा काही आठवडे नंतर परत येते

सारा एक **नवीन थ्रेड** सुरू करते (नवीन सत्राची नक्कल करत आहे). कार्यशील स्मृती रिकामी आहे, पण दीर्घकालीन प्राधान्य संच तिची माहिती अजूनही ठेवून आहे. एजंटने ती प्राप्त करुन वैयक्तिक शिफारसींसाठी वापरावी.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

या धड्यात तुम्ही एजंट मेमरीच्या तीन प्रकारांबद्दल आणि त्यांना Microsoft Agent Framework सह कसे अमलात आणायचे ते शिकलात:

| मेमरी प्रकार | MAF मेकॅनिझम | आयुष्यकाल |
|---|---|---|
| **वर्किंग** | `agent.create_session()` | एकल संभाषण |
| **शॉर्ट-टर्म** | थ्रेडमधील संचयित संदर्भ | एकल कार्य / सत्र |
| **लाँग-टर्म** | `@tool` फंक्शन्सद्वारे प्रवेश केलेला बाह्य संग्रह | सत्रांमध्ये |

### मुख्य मुद्दे
1. **`agent.create_session()`** वर्किंग मेमरी प्रदान करते — एजंट सत्रांतील पूर्ण संभाषण इतिहास पाहू शकतो.
2. **नवीन सत्रे संदर्भ गमावतात** — लाँग-टर्म मेमरीशिवाय एजंट मागील संभाषण आठवू शकत नाही.
3. **`@tool` फंक्शन्स** ही खळगी पूर्तता करतात — ते एजंटला माहिती संग्रहित करण्याची आणि पुनर्प्राप्त करण्याची परवानगी देतात.
4. **वैयक्तिकीकरण कालांतराने सुधारते** — जितकी जास्त प्राधान्ये साठवली जातात, तितका एजंटचा सल्ला चांगला होतो.

### प्रत्यक्ष जगातील अनुप्रयोग
- **ग्राहक सेवा**: ग्राहकांचा इतिहास आणि प्राधान्ये लक्षात ठेवणे
- **वैयक्तिक सहाय्यक**: दिवस किंवा आठवड्यांपर्यंत संदर्भ टिकवणे
- **आरोग्य सेवा**: रुग्णांची माहिती आणि प्राधान्ये ट्रॅक करणे
- **ई-कॉमर्स**: इतिहासावर आधारित वैयक्तिकृत खरेदी

### पुढील पावले
- इन-मेमरी डिक्टच्या जागी डेटाबेस किंवा व्हेक्टर स्टोअर वापरा (उदा. Azure AI Search)
- वेळेवर माहिती साठी मेमरीची कालबाह्यता जोडा
- सामायिक मेमरीसह मल्टी-एजंट सिस्टीम तयार करा
- ज्ञान-ग्राफ-बॅकड मेमरीसाठी Cognee नोटबुक एक्सप्लोर करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
